# 真实本地模型：MTP、LNTP 与 Token Entropy

从本地 Hugging Face 因果模型真实生成，并由 `HuggingFaceWhiteBoxBackend` 提取 token 概率与 top-k 分布。

## 真实集成前提

安装：`python3 -m pip install -e ".[whitebox]"`。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_CAUSAL_MODEL: str`：已下载 causal LM 的本地目录。
- `DEVICE: str = 'cpu'`：推理设备。
- `MAX_NEW_TOKENS: int = 48`：最大生成 token 数。
- `TOP_K: int = 5`：每步保留的候选 token 数。

## 步骤 1：配置真实后端

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from hallucination.detection.backends.hf_whitebox import HuggingFaceWhiteBoxBackend
LOCAL_CAUSAL_MODEL = "/absolute/path/to/your/causal-lm"
DEVICE = "cpu"
MAX_NEW_TOKENS = 48
TOP_K = 5
if not Path(LOCAL_CAUSAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_CAUSAL_MODEL to an existing local model directory.")
backend = HuggingFaceWhiteBoxBackend.from_pretrained(LOCAL_CAUSAL_MODEL, device=DEVICE)

## 步骤 2：真实生成并评分

In [ ]:
from hallucination.detection.pipelines.whitebox_generation import WhiteBoxGenerationPipeline
pipeline = WhiteBoxGenerationPipeline(backend, include_entropy=True, threshold=0.5)
result = pipeline.detect_prompt("请用一句中文说明蓝鲸主要吃什么。", max_new_tokens=MAX_NEW_TOKENS, top_k=TOP_K, do_sample=False)
print(result.metadata["response"])
for item in result.method_scores:
    print(item.method, item.score, item.risk, item.raw)

## 结果解读与限制

top-k entropy 是近似值；token 概率衡量生成不确定性，不证明外部事实是否正确。